# Course 3 · Week 1 — Hands-on: K-means and Anomaly Detection

Two unsupervised algorithms in one notebook. They share a worldview — fit something to "what's normal in the data" — but they answer different questions.

By the end you'll have:

1. Implemented K-means (assign + recenter loop) and watched it converge on three blobs
2. Implemented Gaussian density fitting in numpy
3. Built an anomaly detector that flags rare events using a probability threshold
4. (Stretch) Plotted a precision-recall curve as the threshold varies

Estimated time: **45–60 minutes.**


## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Three Gaussian blobs in 2D — but we won't tell K-means the labels.
np.random.seed(2)
m_per_cluster = 40
centers_true = np.array([[-2, -2], [2, -2], [0, 2.5]])
X_clusters = np.vstack([np.random.randn(m_per_cluster, 2) * 0.6 + c for c in centers_true])
print(f"X shape: {X_clusters.shape}")

K = 3


In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(X_clusters[:, 0], X_clusters[:, 1], color="gray", s=40, alpha=0.6)
plt.xlabel("$x_1$"); plt.ylabel("$x_2$")
plt.title("Unlabeled data — find the clusters")
plt.grid(alpha=0.3); plt.axis("equal")
plt.show()


## Quick recap — K-means

The simplest unsupervised algorithm. You pick `K` (the number of clusters you want), and the algorithm:

1. **Assign** each point to the nearest centroid
2. **Update** each centroid to be the mean of its assigned points
3. Repeat until nothing changes

Two clean facts:
- The cost `J` (mean squared distance from each point to its centroid) **never increases** between iterations. Each step solves one half of the optimization while holding the other fixed.
- The result depends on initialization. Bad starts → bad local minima. Standard trick: run K-means with several random starts, pick the best.


## Exercise 1 — assign each point to its nearest centroid

In [ ]:
def assign_clusters(X, centroids):
    """For each point in X, return the index of the nearest centroid.

    X:         shape (m, n)
    centroids: shape (K, n)
    Returns:   shape (m,)  — integer indices in [0, K)
    """
    # TODO: compute distances from every point to every centroid (shape (m, K)),
    #       then take argmin along axis=1
    return np.zeros(len(X), dtype=int)


# Sanity: 3 points, 2 centroids
X_test = np.array([[0, 0], [1, 1], [10, 10]])
centroids_test = np.array([[0.5, 0.5], [9, 9]])
out = assign_clusters(X_test, centroids_test)
print(f"out = {out}   expected [0, 0, 1]")
assert (out == np.array([0, 0, 1])).all()
print("✓ assign_clusters() works")


## Exercise 2 — recenter each centroid

In [ ]:
def update_centroids(X, c, K):
    """Recompute each centroid as the mean of its assigned points.

    Edge case: if a cluster ends up empty, leave its centroid unchanged.
    """
    new_centroids = np.zeros((K, X.shape[1]))
    for k in range(K):
        # TODO: compute mean of points assigned to cluster k (if any)
        pass
    return new_centroids


# Sanity check
c_test = np.array([0, 0, 1])
new_c = update_centroids(X_test, c_test, 2)
print(f"new centroids:\n{new_c}")
print(f"expected: [[0.5, 0.5], [10, 10]]")
assert abs(new_c[0, 0] - 0.5) < 1e-9
assert abs(new_c[1, 0] - 10.0) < 1e-9
print("✓ update_centroids() works")


## Exercise 3 — distortion (the K-means cost)

In [ ]:
def distortion(X, c, centroids):
    """Mean squared distance from each point to its assigned centroid (the K-means cost).

    Math: J = (1/m) * sum_i ||x_i - centroid[c_i]||^2
    """
    # TODO: implement
    return 0.0


# Sanity: if every point is at its centroid, distortion is 0
X_perfect = np.array([[0, 0], [10, 10]])
c_perf = np.array([0, 1])
cents_perf = np.array([[0, 0], [10, 10]])
assert distortion(X_perfect, c_perf, cents_perf) == 0.0
print("✓ distortion() works")


## Exercise 4 — full K-means loop

In [ ]:
# Run K-means: alternate assign and update until converged.
np.random.seed(0)
init_idx = np.random.choice(len(X_clusters), K, replace=False)
centroids = X_clusters[init_idx].copy()
print(f"Initial centroids:\n{centroids}")

J_history = []
n_iters = 20
for step in range(n_iters):
    # TODO: assign, then update, then compute distortion
    c = np.zeros(len(X_clusters), dtype=int)  # placeholder
    J = 0.0
    J_history.append(J)
    if step < 5 or step == n_iters - 1:
        print(f"step {step:2d}: J = {J:.4f}")

# Plot final clustering and J curve
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
colors = ["#ef4444", "#10b981", "#3b82f6"]
for k in range(K):
    axes[0].scatter(X_clusters[c == k, 0], X_clusters[c == k, 1], color=colors[k], s=40, alpha=0.7, label=f"cluster {k}")
axes[0].scatter(centroids[:, 0], centroids[:, 1], color="black", marker="X", s=200, lw=2, label="centroids")
axes[0].set_title("Final clustering"); axes[0].legend(); axes[0].axis("equal"); axes[0].grid(alpha=0.3)

axes[1].plot(J_history, "o-", color="#4338ca")
axes[1].set_xlabel("iteration"); axes[1].set_ylabel("distortion J")
axes[1].set_title("J monotonically decreases"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


## Anomaly detection

In [ ]:
# Anomaly detection setup: 80 "normal" points + 3 obvious anomalies
np.random.seed(7)
m_norm = 80
X_norm = np.random.randn(m_norm, 2) * np.array([1.0, 0.7]) + np.array([0.0, 1.0])
anomalies = np.array([[5.0, 5.0], [-4.5, 4.5], [4.0, -3.0]])
X_anom = np.vstack([X_norm, anomalies])
y_anom = np.array([0]*m_norm + [1]*3)
print(f"X_anom shape: {X_anom.shape}, true anomalies: {int(y_anom.sum())}")

plt.figure(figsize=(6, 5))
plt.scatter(X_anom[y_anom == 0, 0], X_anom[y_anom == 0, 1], color="#3b82f6", s=40, label="normal", alpha=0.7)
plt.scatter(X_anom[y_anom == 1, 0], X_anom[y_anom == 1, 1], color="#ef4444", s=80, marker="X", label="anomaly")
plt.legend(); plt.grid(alpha=0.3); plt.axis("equal"); plt.title("Spot the anomalies")
plt.show()


## Quick recap — anomaly detection

If you can model what "normal" looks like, you can flag anything that doesn't look normal.

Standard approach: fit an independent Gaussian to each feature, multiply the per-feature probabilities to get a joint probability, threshold at `eps`. Below `eps` → anomaly.

This works when normal data clusters in some bounded region and anomalies are far outside. It fails when "normal" is multi-modal (multiple clusters) or when features are strongly correlated. For those cases you'd use multivariate Gaussian or other density models.


## Exercise 5 — fit a Gaussian, compute probabilities

In [ ]:
def fit_gaussian(X):
    """Fit one independent Gaussian per feature.

    Returns: (mu, sigma2) — both shape (n,)
    """
    # TODO: compute per-feature mean and variance
    mu = np.zeros(X.shape[1])
    sigma2 = np.zeros(X.shape[1])
    return mu, sigma2


def p_gaussian(X, mu, sigma2):
    """Joint probability under independent feature Gaussians.

    Math: p(x) = prod_j  (1 / sqrt(2*pi*sigma2_j)) * exp(-(x_j - mu_j)^2 / (2*sigma2_j))
    """
    # TODO: implement (numpy makes this nicely vectorized)
    return np.zeros(len(X))


mu, sigma2 = fit_gaussian(X_norm)
print(f"mu     = {np.round(mu, 3)}     expect [≈0.04, ≈0.93]")
print(f"sigma² = {np.round(sigma2, 3)}  expect [≈0.89, ≈0.50]")
assert abs(mu[0] - 0.038) < 0.05 and abs(mu[1] - 0.933) < 0.05
print("✓ fit_gaussian() works")

p = p_gaussian(X_anom, mu, sigma2)
print(f"\nProbabilities — first 5: {np.round(p[:5], 4)}")
print(f"Last 3 (the anomalies): {np.round(p[-3:], 6)}")
assert (p[-3:] < p[:m_norm].min() / 100).all(), "Anomalies should have much lower probability than normal points"
print("✓ p_gaussian() works")


## Exercise 6 — threshold and flag

In [ ]:
# Pick a threshold and flag anomalies
eps = 1e-5
flagged = p < eps

# Confusion-matrix-ish report
true_positives  = int(((flagged == 1) & (y_anom == 1)).sum())
false_positives = int(((flagged == 1) & (y_anom == 0)).sum())
false_negatives = int(((flagged == 0) & (y_anom == 1)).sum())
print(f"With eps = {eps}:")
print(f"  True positives  (caught real anomalies): {true_positives} / 3")
print(f"  False positives (normal flagged):        {false_positives}")
print(f"  False negatives (missed anomalies):      {false_negatives}")


## ⭐ Stretch — precision-recall curve

In [ ]:
# STRETCH: try several thresholds. Plot precision vs recall as eps varies.

def precision_recall(p, y, eps):
    flagged = p < eps
    tp = int(((flagged == 1) & (y == 1)).sum())
    fp = int(((flagged == 1) & (y == 0)).sum())
    fn = int(((flagged == 0) & (y == 1)).sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return precision, recall

eps_grid = np.logspace(-10, -1, 30)
# TODO: for each eps, compute precision and recall, then plot precision-recall curve
